Ejercicio 1

In [ ]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import requests
import re
import io
from urllib.parse import urlencode, urlunsplit

In [ ]:
def get_soup(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content)
    return soup

url = 'https://www.nexford.edu/insights/highest-payingjobs-in-the-world'
soup = get_soup(url)

In [ ]:
def generate_database(soup):
    trabajos = soup.findAll(name='h3')
    salarios = soup.findAll(name='a')
    lista_trabajos = []
    patron_trabajo = re.compile(r'(\d+)\.(\w+)')
    for trabajo in trabajos:
        m = patron_trabajo.search(trabajo)
        if m is not None:
            lista_trabajos.append(m.group(2))

    lista_salarios = []
    patron_salarios = re.compile(r'\$(\d+)\,(\d+)')
    for salario in salarios:
        m = patron_salarios.search(salario)
        if m is not None:
            lista_salarios.append(salario)
    df = pd.DataFrame(salarios,index=trabajos,columns=['salary_average'])
    return df

In [ ]:
def obtain_numeric_salary(df:pd.DataFrame):
    patron_salarios = re.compile(r'\$(\d+)\,(\d+)')
    def cambio(celda):
        m = patron_salarios.search(celda)
        if m is not None:
            entero = m.group(1)
            decimal = m.group(2)
            return float(f'{entero}.{decimal}')
    df['salary_average']=df['salary_average'].apply(cambio)
    return df
df = pd.read_csv('best_jobs_1c.csv',sep=';')
df_scrapeado= obtain_numeric_salary(df)
df_scrapeado

Ejercicio 2

In [ ]:
SCHEME = 'https'
BASE = 'jobicy.com/api/v2'
PATH = '/remote-jobs'
query_dict = {'tag':'data scientist'}

def get_url(path,query_dict= None):
    if query_dict is None:
        query_dict = {}
    query = urlencode(query_dict)
    return urlunsplit((SCHEME,BASE,path,query,''))

url = get_url(PATH,query_dict)
url

In [ ]:
response = requests.get(url)
respuesta_json = response.json()
respuesta_json

In [ ]:
def get_data_scientist(json):
    if json['jobCount']>1:
        trabajos = json['jobs']
        filas = []
        for trabajo in trabajos:
            job = 'Data Scientist'
            industria = trabajo['jobIndustry'][0]
            try:
                salary = trabajo['salaryMin']
            except:
                salary = np.nan
            fila = {'job':job,'industry':industria,'salary':salary}
            filas.append(fila)
        return pd.DataFrame(filas)

    else:
        return 'Solo existe un unico trabajo'

get_data_scientist(respuesta_json)

Ejercicio 3

In [ ]:
df_jobs = pd.read_csv('all_jobs_3a.csv',sep=';')
df_jobs

In [ ]:
def impute_outliers(df_jobs:pd.DataFrame):
    df_jobs['z_score'] = abs((df_jobs['salary'] -df_jobs['salary'].mean())/df_jobs['salary'].std())
    df_jobs.loc[df_jobs['z_score']>3,'salary'] = df_jobs['salary'].median()
    return df_jobs[['job','industry','salary']]
df_out = impute_outliers(df_jobs)
df_out

In [ ]:
def impute_missing(df_out:pd.DataFrame):
    df = df_out.copy()
    df['salary'] = (df_out.groupby('industry')['salary'].transform(lambda m: m.fillna(m.median())))
    return df
df_jobs = impute_missing(df_out)
df_jobs

Ejercicio 4

In [ ]:
def compare_sources(df_scrapeado:pd.DataFrame, df_jobs:pd.DataFrame):
    df = df_scrapeado.merge(df_jobs, on='job')
    df['variation'] = (df['salary']-df['salary_average'])/df['salary']
    return df.groupby('job')['variation'].mean().sort_values(ascending=False).reset_index()

compare_sources(df_scrapeado,df_jobs)